# Value Estimator Variance Analysis

This notebook analyzes the variance of different value estimation methods compared to Monte Carlo.

For each state, method, and training data size (n_episodes), we:
1. Compute mean and variance of predictions across different training batches
2. Calculate variance ratio: Var(method) / Var(MC)
3. Plot histogram of variance ratios for each data size
4. Compare variance ratios across different training data sizes

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Predictions Data

In [3]:
EXPERIMENT_ID = "cartpole_test"
EXPERIMENT_DIR = Path(f"experiments/{EXPERIMENT_ID}")
RESULTS_DIR = EXPERIMENT_DIR / "results"

predictions_file = RESULTS_DIR / "predictions.csv"
if predictions_file.exists():
    df = pd.read_csv(predictions_file)
    print(f"Loaded {len(df)} predictions")
    print(f"\nDataset shape:")
    print(f"  States: {df['state_idx'].nunique()}")
    print(f"  Methods: {df['method'].nunique()}")
    print(f"  Batches: {df['batch_idx'].nunique()}")
    print(f"  Episode subsets: {df['n_episodes'].nunique()}")
    print(f"\nMethods: {sorted(df['method'].unique())}")
    print(f"Episode subsets: {sorted(df['n_episodes'].unique())}")
    print(f"\nFirst few rows:")
    display(df.head(10))
else:
    print(f"ERROR: Predictions file not found at {predictions_file}")
    print("Please run: python -m src.evaluate --config <your_config>.yaml")

ERROR: Predictions file not found at experiments/cartpole_test/results/predictions.csv
Please run: python -m src.evaluate --config <your_config>.yaml


## 2. Compute Mean and Variance per State and Episode Subset

In [ ]:
stats = df.groupby(['state_idx', 'method', 'n_episodes'])['predicted_value'].agg(
    mean='mean',
    variance='var',
    std='std',
    count='count'
).reset_index()

print(f"Computed statistics for {len(stats)} (state, method, n_episodes) tuples")
print(f"\nExample statistics:")
display(stats.head(10))

print(f"\nMissing values:")
print(stats.isnull().sum())

print(f"\nStatistics by episode subset:")
for n_eps in sorted(stats['n_episodes'].unique()):
    n_entries = len(stats[stats['n_episodes'] == n_eps])
    print(f"  {n_eps} episodes: {n_entries} entries")

## 3. General Function for Computing Log Ratios

In [ ]:
def compute_log_ratios(stats_df, stat_column, stat_name, n_episodes=None, save_suffix=''):
    """
    Compute and plot log ratios of a statistic for each method against Monte Carlo.
    
    Args:
        stats_df: DataFrame with columns ['state_idx', 'method', 'n_episodes', stat_column]
        stat_column: Name of the column to compute ratios for
        stat_name: Human-readable name for the statistic
        n_episodes: If specified, filter to this episode subset only
        save_suffix: Suffix to add to saved plot filename
    
    Returns:
        DataFrame with log ratios
    """
    if n_episodes is not None:
        stats_df = stats_df[stats_df['n_episodes'] == n_episodes].copy()
        title_suffix = f" ({n_episodes} episodes)"
    else:
        title_suffix = " (all episode subsets)"
    
    mc_values = stats_df[stats_df['method'] == 'monte_carlo'][['state_idx', 'n_episodes', stat_column]]
    mc_values.columns = ['state_idx', 'n_episodes', 'mc_value']
    
    merged = stats_df.merge(mc_values, on=['state_idx', 'n_episodes'])
    
    merged['log_ratio'] = np.log(merged[stat_column] / merged['mc_value'])
    
    ratios = merged[merged['method'] != 'monte_carlo'].copy()
    
    if len(ratios) == 0:
        print(f"No data to plot for n_episodes={n_episodes}")
        return ratios
    
    fig, ax = plt.subplots(figsize=(12, 6))
    for method in sorted(ratios['method'].unique()):
        method_data = ratios[ratios['method'] == method]['log_ratio']
        ax.hist(method_data, bins=50, alpha=0.6, label=f'{method} (mean={method_data.mean():.3f})', edgecolor='black')
    
    ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Equal to MC')
    ax.set_xlabel(f'Log {stat_name} Ratio: log(Method / MC)', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title(f'{stat_name} Log Ratio Distribution vs Monte Carlo{title_suffix}', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    
    plot_file = RESULTS_DIR / f'{stat_column}_log_ratio{save_suffix}.png'
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    print(f"Saved plot to {plot_file}")
    plt.show()
    
    return ratios

print("Function defined: compute_log_ratios()")

## 3. Variance Analysis by Episode Subset

In [ ]:
episode_subsets = sorted(stats['n_episodes'].unique())
print(f"Analyzing variance ratios for {len(episode_subsets)} episode subsets: {episode_subsets}\n")

variance_ratios_by_subset = {}
for n_eps in episode_subsets:
    print(f"\n{'='*80}")
    print(f"Variance analysis for {n_eps} episodes")
    print(f"{'='*80}")
    ratios = compute_log_ratios(stats, 'variance', 'Variance', n_episodes=n_eps, save_suffix=f'_{n_eps}_episodes')
    variance_ratios_by_subset[n_eps] = ratios

In [1]:
## 4. Mean Analysis by Episode Subset

In [ ]:
mean_ratios_by_subset = {}
for n_eps in episode_subsets:
    print(f"\n{'='*80}")
    print(f"Mean analysis for {n_eps} episodes")
    print(f"{'='*80}")
    ratios = compute_log_ratios(stats, 'mean', 'Mean', n_episodes=n_eps, save_suffix=f'_{n_eps}_episodes')
    mean_ratios_by_subset[n_eps] = ratios

## 5. Comparative Analysis Across Episode Subsets

### 5.1 Variance Ratio Comparison Across Training Data Sizes

In [ ]:
fig, axes = plt.subplots(1, len(episode_subsets), figsize=(6*len(episode_subsets), 5), sharey=True)
if len(episode_subsets) == 1:
    axes = [axes]

for idx, n_eps in enumerate(episode_subsets):
    ratios = variance_ratios_by_subset[n_eps]
    ax = axes[idx]
    
    for method in sorted(ratios['method'].unique()):
        method_data = ratios[ratios['method'] == method]['log_ratio']
        ax.hist(method_data, bins=40, alpha=0.6, label=f'{method} (μ={method_data.mean():.3f})', edgecolor='black')
    
    ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Equal to MC')
    ax.set_xlabel(f'Log Variance Ratio', fontsize=11)
    if idx == 0:
        ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{n_eps} Episodes', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Variance Log Ratio Comparison Across Training Data Sizes', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

plot_file = RESULTS_DIR / 'variance_log_ratio_comparison.png'
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
print(f"Saved plot to {plot_file}")
plt.show()

### 5.2 Mean Ratio Comparison Across Training Data Sizes

In [ ]:
fig, axes = plt.subplots(1, len(episode_subsets), figsize=(6*len(episode_subsets), 5), sharey=True)
if len(episode_subsets) == 1:
    axes = [axes]

for idx, n_eps in enumerate(episode_subsets):
    ratios = mean_ratios_by_subset[n_eps]
    ax = axes[idx]
    
    for method in sorted(ratios['method'].unique()):
        method_data = ratios[ratios['method'] == method]['log_ratio']
        ax.hist(method_data, bins=40, alpha=0.6, label=f'{method} (μ={method_data.mean():.3f})', edgecolor='black')
    
    ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Equal to MC')
    ax.set_xlabel(f'Log Mean Ratio', fontsize=11)
    if idx == 0:
        ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{n_eps} Episodes', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Mean Log Ratio Comparison Across Training Data Sizes', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

plot_file = RESULTS_DIR / 'mean_log_ratio_comparison.png'
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
print(f"Saved plot to {plot_file}")
plt.show()

### 5.3 Summary: Mean Log Ratios vs Training Data Size

In [ ]:
summary_data = []

for n_eps in episode_subsets:
    var_ratios = variance_ratios_by_subset[n_eps]
    mean_ratios = mean_ratios_by_subset[n_eps]
    
    for method in sorted(var_ratios['method'].unique()):
        var_method = var_ratios[var_ratios['method'] == method]['log_ratio']
        mean_method = mean_ratios[mean_ratios['method'] == method]['log_ratio']
        
        summary_data.append({
            'n_episodes': n_eps,
            'method': method,
            'variance_log_ratio_mean': var_method.mean(),
            'variance_log_ratio_std': var_method.std(),
            'mean_log_ratio_mean': mean_method.mean(),
            'mean_log_ratio_std': mean_method.std(),
        })

summary_df = pd.DataFrame(summary_data)
print("Summary of log ratios by method and training data size:")
display(summary_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for method in sorted(summary_df['method'].unique()):
    method_data = summary_df[summary_df['method'] == method]
    ax1.errorbar(method_data['n_episodes'], method_data['variance_log_ratio_mean'], 
                 yerr=method_data['variance_log_ratio_std'], label=method, marker='o', capsize=5)
    ax2.errorbar(method_data['n_episodes'], method_data['mean_log_ratio_mean'], 
                 yerr=method_data['mean_log_ratio_std'], label=method, marker='o', capsize=5)

ax1.axhline(0, color='red', linestyle='--', linewidth=2, alpha=0.5)
ax1.set_xlabel('Training Episodes', fontsize=12)
ax1.set_ylabel('Mean Log Variance Ratio', fontsize=12)
ax1.set_title('Variance Ratio vs Training Data Size', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.axhline(0, color='red', linestyle='--', linewidth=2, alpha=0.5)
ax2.set_xlabel('Training Episodes', fontsize=12)
ax2.set_ylabel('Mean Log Mean Ratio', fontsize=12)
ax2.set_title('Mean Ratio vs Training Data Size', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()

plot_file = RESULTS_DIR / 'ratio_summary_vs_data_size.png'
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
print(f"\nSaved plot to {plot_file}")
plt.show()